In [ ]:
A company produces three products: A, B, and C over 3 months: January, February, and March.

The company must decide how many units of each product to produce in each month while satisfying monthly demand and respecting production capacity.

Products can be produced before they are needed and stored as inventory for future months.

Product Data
Product	Profit per unit (€)	Machine hours per unit
A	40	2
B	55	3
C	70	4
Monthly Production Capacity
Month	Available machine hours
January	230
February	250
March	255
Monthly Demand
Month	A	B	C
January	30	20	15
February	40	30	20
March	50	35	25
Inventory Rules
Initial inventory of A, B, and C is 0.
Products can be produced in advance and stored for future months.
Inventory holding cost is €5 per unit per month.
There is no backlogging: each month's demand must be satisfied during that month.
Inventory cannot be negative.
Inventory at the end of March must be 0.


Questions
How many units of A, B, and C should be produced in each month?
How many units of each product should be carried as inventory from one month to the next?
How many machine hours are used in each month?
How much machine capacity remains unused in each month?
What is the total production profit?
What is the total inventory cost?
What is the maximum total profit after inventory costs?

SyntaxError: invalid character '€' (U+20AC) (95174073.py, line 8)

In [ ]:
import pandas as pd
P=pd.read_excel("P.xlsx")
M=pd.read_excel("M.xlsx")
R=pd.read_excel("R.xlsx")
print(P)




      Month Product  Demand  Avaliable machine  Profit  Machine hours
0   January       A      30                230      40              2
1   January       B      20                230      55              3
2   January       C      15                230      70              4
3  February       A      40                250      40              2
4  February       B      30                250      55              3
5  February       C      20                250      70              4
6     March       A      50                255      40              2
7     March       B      35                255      55              3
8     March       C      25                255      70              4


In [ ]:
print(P.columns)
P.columns=P.columns.str.strip()
print(P.columns)

Index(['Month', 'Product', 'Demand', 'Avaliable machine', 'Profit',
       'Machine hours'],
      dtype='object')
Index(['Month', 'Product', 'Demand', 'Avaliable machine', 'Profit',
       'Machine hours'],
      dtype='object')


In [ ]:
from ast import Param
import pyomo.environ as pyo
!apt-get update -qq
!apt-get install -y -qq glpk-utils
model=pyo.ConcreteModel()
#sets
model.P = pyo.Set(initialize=P["Product"].tolist())

model.M = pyo.Set(
    initialize=M["Month"].tolist(),
    ordered=True
)

#Parameters
model.Holding_cost = pyo.Param(initialize=5)
model.Profit=pyo.Param(model.P,initialize=P.set_index("Product")["Profit"].to_dict())
model.Machine_hours=pyo.Param(model.P,initialize=P.set_index("Product")["Machine hours"].to_dict())
model.Available_machine=pyo.Param(model.M,initialize=R.set_index("Month")["Available machine"].to_dict())
model.Demand = pyo.Param(
    model.P,
    model.M,
    initialize={
        (p, row["Month"]): row[p]
        for _, row in M.iterrows()
        for p in model.P
    }
)



W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [ ]:
model.x = pyo.Var(
    model.P,
    model.M,
    within=pyo.NonNegativeReals
)

model.I = pyo.Var(
    model.P,
    model.M,
    within=pyo.NonNegativeReals
)
print(model.x)
print(model.I)

x
I


In [ ]:
#Constraint
def machine_capacity_rule(model, m):
    return sum(
        model.x[p, m] * model.Machine_hours[p]
        for p in model.P
    ) <= model.Available_machine[m]

model.MachineCapacity = pyo.Constraint(
    model.M,
    rule=machine_capacity_rule
)


This means:

Production × machine hours ≤ available machine hours.

For January, Pyomo creates:

$$ 2x_{A,Jan}+3x_{B,Jan}+4x_{C,Jan}\le230 $$

In [ ]:
def inventory_balance_rule(model, p, m):

    if m == model.M.first():
        return model.x[p, m] == model.Demand[p, m] + model.I[p, m]

    else:
        previous_m = model.M.prev(m)

        return (
            model.I[p, previous_m] + model.x[p, m]
            == model.Demand[p, m] + model.I[p, m]
        )


model.InventoryBalance = pyo.Constraint(
    model.P,
    model.M,
    rule=inventory_balance_rule
)

def final_inventory_rule(model, p):
    return model.I[p, model.M.last()] == 0

model.FinalInventory = pyo.Constraint(
    model.P,
    rule=final_inventory_rule
)

In [ ]:
def objective_rule(model):
    return (
        sum(
            model.Profit[p] * model.x[p, m]
            for p in model.P
            for m in model.M
        )
        -
        model.Holding_cost * sum(
            model.I[p, m]
            for p in model.P
            for m in model.M
        )
    )

model.Objective = pyo.Objective(
    rule=objective_rule,
    sense=pyo.maximize
)

In [ ]:
print(model.Objective)
print(model.MachineCapacity)
print(model.InventoryBalance)
print(model.FinalInventory)


Objective
MachineCapacity
InventoryBalance
FinalInventory


In [ ]:
solver = pyo.SolverFactory(
    "glpk",
    executable="/usr/bin/glpsol"
)

results = solver.solve(model)

print(results.solver.status)
print(results.solver.termination_condition)

ok
optimal


In [ ]:
print(hasattr(model, "x"))
print(hasattr(model, "I"))


for p in model.P:
    for m in model.M:
        print(p, m, model.x[p,m].value)


True
True
A January 30.0
A February 40.0
A March 50.0
B January 20.0
B February 30.0
B March 35.0
C January 27.5
C February 20.0
C March 12.5
A January 0.0
A February 0.0
A March 0.0
B January 0.0
B February 0.0
B March 0.0
C January 12.5
C February 12.5
C March 0.0


In [ ]:
#Check inventory
for p in model.P:
    for m in model.M:
        print(p, m, model.I[p, m].value)

A January 0.0
A February 0.0
A March 0.0
B January 0.0
B February 0.0
B March 0.0
C January 12.5
C February 12.5
C March 0.0


In [ ]:
import pandas as pd

production_table = []

for p in model.P:
    for m in model.M:
        production_table.append({
            "Product": p,
            "Month": m,
            "Production": pyo.value(model.x[p, m])
        })

production_df = pd.DataFrame(production_table)

production_df

,Product,Month,Production
0,A,January,30.0
1,A,February,40.0
2,A,March,50.0
3,B,January,20.0
4,B,February,30.0
5,B,March,35.0
6,C,January,27.5
7,C,February,20.0
8,C,March,12.5


In [ ]:
inventory_table = []

for p in model.P:
    for m in model.M:
        inventory_table.append({
            "Product": p,
            "Month": m,
            "Inventory": pyo.value(model.I[p, m])
        })

inventory_df = pd.DataFrame(inventory_table)

inventory_df

,Product,Month,Inventory
0,A,January,0.0
1,A,February,0.0
2,A,March,0.0
3,B,January,0.0
4,B,February,0.0
5,B,March,0.0
6,C,January,12.5
7,C,February,12.5
8,C,March,0.0


In [ ]:
import pandas as pd
import pyomo.environ as pyo

results_table = []

for p in model.P:
    for m in model.M:

        production = pyo.value(model.x[p, m])
        demand = pyo.value(model.Demand[p, m])
        inventory = pyo.value(model.I[p, m])
        machine_hours = pyo.value(model.Machine_hours[p])

        results_table.append({
            "Product": p,
            "Month": m,
            "Production": production,
            "Demand": demand,
            "Inventory": inventory,
            "Machine Hours Used": production * machine_hours,
            "Profit": production * pyo.value(model.Profit[p])
        })

results_df = pd.DataFrame(results_table)

results_df

,Product,Month,Production,Demand,Inventory,Machine Hours Used,Profit
0,A,January,30.0,30,0.0,60.0,1200.0
1,A,February,40.0,40,0.0,80.0,1600.0
2,A,March,50.0,50,0.0,100.0,2000.0
3,B,January,20.0,20,0.0,60.0,1100.0
4,B,February,30.0,30,0.0,90.0,1650.0
5,B,March,35.0,35,0.0,105.0,1925.0
6,C,January,27.5,15,12.5,110.0,1925.0
7,C,February,20.0,20,12.5,80.0,1400.0
8,C,March,12.5,25,0.0,50.0,875.0
